**IMPORTING ALL MODULES**

In [10]:
import numpy as np
import pandas as pd
import ast
import pickle
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

**1. DATA LOADING & MERGING**

In [11]:
# Load the datasets
raw_movies = pd.read_csv('./sample_data/tmdb_5000_movies.csv')
raw_credits = pd.read_csv('./sample_data/tmdb_5000_credits.csv')

# Merge datasets on the 'title' column to combine metadata with cast/crew info
movies_df = pd.merge(raw_movies, raw_credits, on='title', how='outer')

# Select only the relevant features for a content-based recommender
movies_df = movies_df[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

# Remove rows with missing values (specifically in the 'overview' column)
movies_df.dropna(inplace=True)
movies_df.sample(5)

,movie_id,title,overview,genres,keywords,cast,crew
187,246449,Against the Wild,The action-packed feature film tells the drama...,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 10751...","[{""id"": 1964, ""name"": ""cave""}, {""id"": 11116, ""...","[{""cast_id"": 1, ""character"": ""Susan Wade"", ""cr...","[{""credit_id"": ""52fe4f14c3a36847f82bc027"", ""de..."
1349,37495,Four Lions,Four Lions tells the story of a group of Briti...,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 80, ""nam...","[{""id"": 13015, ""name"": ""terrorism""}, {""id"": 19...","[{""cast_id"": 5, ""character"": ""Omar"", ""credit_i...","[{""credit_id"": ""536befdfc3a368124100a6e6"", ""de..."
1683,2026,Hostage,When a mafia accountant is taken hostage on hi...,"[{""id"": 9648, ""name"": ""Mystery""}, {""id"": 18, ""...","[{""id"": 1812, ""name"": ""fbi""}, {""id"": 1930, ""na...","[{""cast_id"": 12, ""character"": ""Jeff Talley"", ""...","[{""credit_id"": ""52fe432ec3a36847f8040603"", ""de..."
716,9759,Cellular,A young man receives an emergency phone call o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 974, ""name"": ""bank""}, {""id"": 1880, ""na...","[{""cast_id"": 20, ""character"": ""Ryan Hewitt"", ""...","[{""credit_id"": ""5693a8dcc3a3684d04004f4f"", ""de..."
2367,612,Munich,"During the 1972 Olympic Games in Munich, eleve...","[{""id"": 18, ""name"": ""Drama""}, {""id"": 28, ""name...","[{""id"": 90, ""name"": ""paris""}, {""id"": 441, ""nam...","[{""cast_id"": 10, ""character"": ""Avner"", ""credit...","[{""credit_id"": ""52fe425dc3a36847f8018a71"", ""de..."


**2. DATA PREPROCESSING & CLEANING**

In [12]:
def extract_names_from_json(json_text):
    """Extracts 'name' values from JSON-like string lists (Genres, Keywords)."""
    names_list = []
    for item in ast.literal_eval(json_text):
        names_list.append(item['name'])
    return names_list

def extract_top_cast(json_text):
    """Extracts the names of the top 3 actors from the cast list."""
    cast_list = []
    counter = 0
    for item in ast.literal_eval(json_text):
        if counter < 3:
            cast_list.append(item['name'])
        counter += 1
    return cast_list

def extract_director(json_text):
    """Extracts the name of the Director from the crew list."""
    directors = []
    for item in ast.literal_eval(json_text):
        if item['job'] == 'Director':
            directors.append(item['name'])
    return directors

def remove_spaces(input_list):
    """Removes spaces from strings to ensure names like 'Johnny Depp' become 'JohnnyDepp'."""
    return [item.replace(" ", "") for item in input_list]

# Apply extraction functions to relevant columns
movies_df['genres'] = movies_df['genres'].apply(extract_names_from_json)
movies_df['keywords'] = movies_df['keywords'].apply(extract_names_from_json)
movies_df['cast'] = movies_df['cast'].apply(extract_top_cast)
movies_df['crew'] = movies_df['crew'].apply(extract_director)

# Format the 'overview' column into a list of words for easier concatenation
movies_df['overview'] = movies_df['overview'].apply(lambda x: x.split())

# Apply space removal to prevent the vectorizer from splitting first and last names
movies_df['cast'] = movies_df['cast'].apply(remove_spaces)
movies_df['crew'] = movies_df['crew'].apply(remove_spaces)
movies_df['genres'] = movies_df['genres'].apply(remove_spaces)
movies_df['keywords'] = movies_df['keywords'].apply(remove_spaces)

movies_df.sample(5)

,movie_id,title,overview,genres,keywords,cast,crew
2748,39514,RED,"[When, his, peaceful, life, is, threatened, by...","[Action, Adventure, Comedy, Crime, Thriller]","[cia, retirement, shottodeath, sniperrifle, re...","[BruceWillis, JohnMalkovich, HelenMirren]",[RobertSchwentke]
2932,51995,Salvation Boulevard,"[Set, in, the, world, of, mega-churches, in, w...","[Comedy, Thriller, Action, Drama]","[pastor, churchservice, spirituality, religion]","[JenniferConnelly, MarisaTomei, PierceBrosnan]",[GeorgeRatliff]
2,333371,10 Cloverfield Lane,"[After, a, car, accident,, Michelle, awakens, ...","[Thriller, ScienceFiction, Drama]","[kidnapping, bunker, paranoia, basement, survi...","[MaryElizabethWinstead, JohnGoodman, JohnGalla...",[DanTrachtenberg]
1836,5915,Into the Wild,"[The, true, story, of, top, student, and, athl...","[Adventure, Drama]","[malenudity, parentskidsrelationship, camping,...","[EmileHirsch, MarciaGayHarden, WilliamHurt]",[SeanPenn]
3678,9566,The Fan,"[When, the, San, Francisco, Giants, pay, cente...","[Drama, Mystery, Thriller]","[sport, luck, sanfranciscogiants, childcustody...","[RobertDeNiro, WesleySnipes, EllenBarkin]",[TonyScott]


**3.CREATING THE TAGS FOR RECOMMENDATION**

In [13]:
# Combine all descriptive features into a single 'tags' column
movies_df['tags'] = (movies_df['overview'] + movies_df['genres'] +
                    movies_df['keywords'] + movies_df['cast'] +
                    movies_df['crew'])

# Create a simplified dataframe with the final tags
movies_df_final = movies_df.drop(columns=['overview', 'genres', 'keywords', 'cast', 'crew'])

# Convert the list of tags back into a single string for Vectorization
movies_df_final['tags'] = movies_df_final['tags'].apply(lambda x: " ".join(x).lower())

movies_df.sample(5)

,movie_id,title,overview,genres,keywords,cast,crew,tags
3715,29078,The Game of Their Lives,"[Based, on, a, true, story,, this, film, tells...",[Drama],[sport],"[GerardButler, WesBentley, GavinRossdale]",[DavidAnspaugh],"[Based, on, a, true, story,, this, film, tells..."
4076,23570,The Pallbearer,"[Aspiring, architect, Tom, Thompson, is, told,...","[Comedy, Romance]","[independentfilm, mistakenidentity]","[DavidSchwimmer, GwynethPaltrow, MichaelRapaport]",[MattReeves],"[Aspiring, architect, Tom, Thompson, is, told,..."
4450,11165,Tora! Tora! Tora!,"[In, the, summer, of, 1941,, the, United, Stat...","[History, Action, Drama, Adventure, War]","[japan, worldwarii, pearlharbor, soldier, impe...","[MartinBalsam, SôYamamura, JosephCotten]","[RichardFleischer, KinjiFukasaku, ToshioMasuda]","[In, the, summer, of, 1941,, the, United, Stat..."
291,9360,Anaconda,"[A, ""National, Geographic"", film, crew, is, ta...","[Adventure, Horror, Thriller]","[amazon, jungle, anaconda, filmcrew, killersna...","[JenniferLopez, IceCube, JonVoight]",[LuisLlosa],"[A, ""National, Geographic"", film, crew, is, ta..."
3881,16888,The Ladies Man,"[Because, of, his, salacious, language,, late-...",[Comedy],"[femalenudity, tattoo, radio, cheatonhusband, ...","[TimMeadows, KarynParsons, BillyDeeWilliams]",[ReginaldHudlin],"[Because, of, his, salacious, language,, late-..."


**4. VECTORIZATION & SIMILARITY CALCULATION**

In [14]:
# Convert tags into numerical vectors using Bag of Words (CountVectorizer)
# We limit to 5000 features and remove common English 'stop words'
cv = CountVectorizer(max_features=5000, stop_words='english')
tag_vectors = cv.fit_transform(movies_df_final['tags']).toarray()

# Calculate Cosine Similarity between all movie vectors
similarity_matrix = cosine_similarity(tag_vectors)

similarity_matrix

array([[1.        , 0.0496904 , 0.05063697, ..., 0.03131121, 0.08164966,
        0.        ],
       [0.0496904 , 1.        , 0.01887128, ..., 0.        , 0.02028602,
        0.        ],
       [0.05063697, 0.01887128, 1.        , ..., 0.07134772, 0.04134491,
        0.07032108],
       ...,
       [0.03131121, 0.        , 0.07134772, ..., 1.        , 0.1789585 ,
        0.20291986],
       [0.08164966, 0.02028602, 0.04134491, ..., 0.1789585 , 1.        ,
        0.05039526],
       [0.        , 0.        , 0.07032108, ..., 0.20291986, 0.05039526,
        1.        ]])

**5. RECOMMENDATION ENGINE**

In [15]:
def recommend_movies(movie_title):
    """Finds and prints the top 5 most similar movies based on cosine similarity."""
    try:
        # Get the index of the movie that matches the title
        movie_index = movies_df_final[movies_df_final['title'] == movie_title].index[0]

        # Get the similarity scores and sort them (skipping the first one as it is the movie itself)
        distances = similarity_matrix[movie_index]
        top_matches = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

        print(f"Movies similar to '{movie_title}':")
        for i in top_matches:
            print(f"- {movies_df_final.iloc[i[0]].title}")

    except IndexError:
        print("Movie not found in the database.")

# Test the recommender
recommend_movies('Avatar')

Movies similar to 'Avatar':
- Titan A.E.
- Small Soldiers
- Ender's Game
- Aliens vs Predator: Requiem
- Independence Day


**6. EXPORTING MODELS**

In [16]:
# Save the final dataframe and similarity matrix for use in a Streamlit app
pickle.dump(movies_df_final, open('movie_list.pkl', 'wb'))
pickle.dump(similarity_matrix, open('similarity.pkl', 'wb'))